# 手動拆解文字生成：分詞 → 生成 → 還原

## 這份程式要做什麼

載入 Phi-3-mini 指令模型，丟給它一段提示詞，讓它接著往下寫。**和 `pipeline` 的差別**是：這裡把背後的三個步驟手動拆開，每一步的中間結果你都看得到。

```
文字提示詞
   │ ① 分詞：變成 token id（張量），搬到 GPU
   ▼
token id
   │ ② 生成：模型逐字產生新的 token id
   ▼
token id（提示詞 + 新生成的部分）
   │ ③ 還原：變回文字
   ▼
文字
```

## 為什麼要學手動拆解

| | `pipeline` | **手動拆解（這份）** |
|---|---|---|
| 程式量 | 很少 | 稍多 |
| 看得到的中間結果 | 看不到 | **看得到每一步** |
| 可控制的細節 | 受限於 pipeline 提供的參數 | **可以自由調整每一步** |
| 適合 | 快速使用 | **理解原理、除錯、客製化** |

學會手動拆解，你就知道 `pipeline` 背後到底發生了什麼，出問題時也知道該檢查哪一步。

## 先搞懂三個關鍵詞

### 1. token

文字不能直接給模型，要先切成小單位（token），再換成數字。一個 token 可能是一個字、半個字，或一個符號。

### 2. 逐字生成

這類模型每次只做一件事：**根據前面所有的 token，預測下一個 token**。生成一整段文字，就是不斷重複：

```
輸入：  Write an email apologizing
預測：  to        → Write an email apologizing to
預測：  Jim       → Write an email apologizing to Jim
...直到產生結束符號，或達到字數上限（max_new_tokens）
```

### 3. 指令模型（instruct）

`Phi-3-mini-4k-instruct` 是在基礎模型上做過指令微調的版本，所以能依指示回答。名稱裡的 `4k` 是上下文長度約 4096 個 token，`mini` 是小型版（我印象中約 38 億參數）。

## 步驟 0：載入套件

| 匯入 | 用途 |
|---|---|
| `AutoModelForCausalLM` | 自動載入因果語言模型（逐字預測下一個詞） |
| `AutoTokenizer` | 自動載入配對的分詞器 |

這份沒有用 `pipeline`，因為三個步驟都自己寫。

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

## 步驟 1：載入模型

```python
model_name = "microsoft/Phi-3-mini-4k-instruct"
device = "cuda"
```

先把兩個會重複使用的值存成變數：

| 變數 | 用在哪裡 |
|---|---|
| `model_name` | 載入模型**和**載入分詞器（確保兩者配對） |
| `device` | 載入模型**和**把輸入張量搬到 GPU（確保兩者在同一個裝置） |

只寫一次，之後要換模型或換裝置，改一個地方就好。

```python
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map = device,
    torch_dtype = "auto",
)
```

| 參數 | 說明 |
|---|---|
| `device_map = device` | 把整個模型放到 GPU（`"cuda"`） |
| `torch_dtype = "auto"` | 依模型設定檔記錄的精度載入（通常是 16 位元），比預設的 32 位元省一半記憶體 |

第一次執行會自動下載模型（數 GB），之後從本機快取讀取。

### ⚠️ 顯存提醒

3.8B 參數以 16 位元載入，權重本身就約 7.6 GB（我用參數量乘上每個參數的位元組數估算的）。**8 GB 的顯卡可能放不下**，容易出現 `CUDA out of memory`。解法：改用 4 位元量化載入（實驗區有範例），或把 `device_map` 改成 `"auto"`（但要注意步驟 4 的 `.to(device)`，見排查表）。

> 版本提醒：在較新版本的 transformers 裡，`torch_dtype` 可能改名為 `dtype`，舊名稱可能會出現警告。我沒有針對你的版本確認。

In [ ]:
model_name = "microsoft/Phi-3-mini-4k-instruct"
device = "cuda"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map = device,
    torch_dtype = "auto",
)

## 步驟 2：載入分詞器

```python
tokenizer = AutoTokenizer.from_pretrained(model_name)
```

分詞器負責**文字和數字之間的轉換**：

```
"Write an email"  ──分詞──▶  [token id, token id, ...]  ──模型──▶  [新的 token id, ...]  ──還原──▶  文字
```

**模型和分詞器必須配對**，所以兩邊都用同一個 `model_name`。用錯配對，數字對不上詞彙表，會得到亂碼或很差的結果。

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name) 

## 步驟 3：準備提示詞

```python
prompt = """
    Write an email apologizing to Jim for the trajic gardening mishap. 
    Explain how it happened. <|assistant|>
    """
```

結尾的 `<|assistant|>` 是 Phi-3 的角色標記，意思是「現在輪到助理回答了」，模型看到它就知道該開始寫回答。

### 三重引號字串會保留所有空白

三重引號 `"""..."""` 會**原封不動保留裡面所有的換行和空白**，所以這個提示詞實際上是：

```
"\n    Write an email ... mishap. \n    Explain how it happened. <|assistant|>\n    "
```

開頭有換行、每行前面有 4 個空格、結尾還有一串空白。這些空白也會被分詞，成為模型看到的輸入的一部分。影響通常不大，但這不是你想要的乾淨輸入。想避免，可以寫成單行字串，或用 `textwrap.dedent` 去掉縮排（見實驗區）。

### 提示詞格式不完整

這個提示詞只有結尾的 `<|assistant|>`，沒有 `<|user|>` 和 `<|end|>`。較穩妥的做法是用 `tokenizer.apply_chat_template`，由分詞器依模型的對話模板產生正確格式（見實驗區）。

> 提示詞裡的 `trajic` 看起來是 `tragic` 的筆誤，我保留原樣。

In [ ]:
prompt = """
    Write an email apologizing to Jim for the trajic gardening mishap. 
    Explain how it happened. <|assistant|>
    """

## 步驟 4：分詞（文字 → token id）

```python
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
```

拆解這一行：

| 部分 | 說明 |
|---|---|
| `tokenizer(prompt, ...)` | 把文字切成 token 並轉成數字 |
| `return_tensors="pt"` | 回傳 **PyTorch 張量**（`pt`），而不是 Python list |
| `.input_ids` | 取出分詞結果裡的 token id（分詞器還會回傳 `attention_mask` 等，這裡只取 id） |
| `.to(device)` | 把張量搬到**和模型相同的裝置**。模型在 GPU，輸入也必須在 GPU，否則會報「裝置不一致」的錯誤 |

張量的形狀是 `(1, N)`：1 筆提示詞、N 個 token（我用常見的情況說明，沒有實際印出，建議自己 `print(input_ids.shape)` 看看）。

In [ ]:
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

## 步驟 5：模型生成（token id → 新的 token id）

```python
generation_output = model.generate(
    input_ids = input_ids,
    max_new_tokens=20
)
```

| 部分 | 說明 |
|---|---|
| `model.generate(...)` | Hugging Face 的**生成函式**：模型逐字預測下一個 token，重複直到遇到結束符號或達到上限 |
| `input_ids = input_ids` | 傳入剛剛分詞的結果 |
| `max_new_tokens=20` | **最多生成 20 個新 token**。這個值很小，一封信遠不止 20 個 token，所以輸出會寫到一半就停 |

### 回傳的是什麼

`generation_output` 是一個張量，形狀是 `(1, N + 新生成的 token 數)`：**包含你輸入的提示詞部分**，再加上新生成的部分。

```
generation_output = [ 提示詞的 token ...  |  新生成的 token ... ]
```

### `max_new_tokens` 與 `max_length`

| 參數 | 計算範圍 |
|---|---|
| `max_new_tokens` | 只算**新生成**的部分 |
| `max_length` | 算**提示詞 + 生成**的總長度 |

用 `max_new_tokens` 比較直觀。

### 生成策略

這裡沒有指定 `do_sample`，所以使用模型的預設生成設定。我印象中通常是貪婪解碼（每一步選機率最高的字，結果固定），但沒有針對你的版本確認。

In [ ]:
generation_output = model.generate(
    input_ids = input_ids,
    max_new_tokens=20
)

## 步驟 6：還原成文字並印出

```python
response = tokenizer.decode(generation_output[0])
print(response)
```

| 部分 | 說明 |
|---|---|
| `generation_output[0]` | 取第一筆（這裡也只有一筆）的 token id |
| `tokenizer.decode(...)` | 把 token id 還原成文字 |

### 為什麼輸出裡有提示詞

因為 `generation_output` 本來就包含提示詞，所以還原出來的文字是「**提示詞 + 生成的內容**」，而且特殊符號（例如 `<|assistant|>`、結束符號）也會一併顯示。

### 怎麼解讀輸出

| 現象 | 原因 |
|---|---|
| 輸出裡**看到你的提示詞** | `generate` 的回傳包含輸入 |
| 生成的信**寫到一半就停了** | 只允許 `max_new_tokens=20` |
| 看到特殊符號 | `decode` 預設不隱藏 |

想看完整的信，把 `max_new_tokens` 調大（例如 200）。具體的輸出內容請以你跑出來的為準。

In [ ]:
response = tokenizer.decode(generation_output[0])
print(response)

---

# 重點整理

## 一、整份程式一句話

> 載入 Phi-3-mini 與其分詞器，把提示詞分詞成 token id 搬到 GPU，用 `model.generate` 逐字生成最多 20 個新 token，再把結果還原成文字印出。

## 二、文字生成的底層流程（必須搞懂）

| 步驟 | 程式 | 輸入 → 輸出 |
|---|---|---|
| ① 分詞 | `tokenizer(prompt, return_tensors="pt").input_ids.to(device)` | 文字 → token id 張量（在 GPU） |
| ② 生成 | `model.generate(input_ids=..., max_new_tokens=20)` | token id → token id（含提示詞） |
| ③ 還原 | `tokenizer.decode(generation_output[0])` | token id → 文字 |

這正是 `pipeline("text-generation")` 在背後做的事。

## 三、必須搞懂的 10 個概念

| # | 概念 | 一句話解釋 |
|---|---|---|
| 1 | token | 文字被切成的小單位，模型處理的基本單位 |
| 2 | 分詞器 | 文字與數字之間的轉換，必須和模型配對 |
| 3 | `return_tensors="pt"` | 讓分詞器回傳 PyTorch 張量 |
| 4 | `.to(device)` | 把張量搬到和模型相同的裝置，否則報錯 |
| 5 | `model.generate` | Hugging Face 的逐字生成函式 |
| 6 | `max_new_tokens` | 最多生成多少個**新** token |
| 7 | 輸出包含提示詞 | `generate` 回傳「輸入 + 新生成」，需要時要自己切掉 |
| 8 | `decode` | token id 還原成文字，預設不隱藏特殊符號 |
| 9 | `torch_dtype="auto"` | 用模型原本記錄的精度載入，省記憶體 |
| 10 | 指令模型 | 做過指令微調，能依指示回答 |

## 四、程式流程速記

```
1. 載入模型          → model_name、device，16 位元放到 GPU
2. 載入分詞器        → 和模型同一個 model_name
3. 準備 prompt       → 結尾的 <|assistant|> 告訴模型該回答了
4. 分詞              → token id 張量，搬到 GPU
5. 生成              → model.generate，最多 20 個新 token
6. 還原並印出        → decode(generation_output[0])
```

## 五、最容易搞混的地方

1. **為什麼輸出裡有我的提示詞？**
   `model.generate` 回傳的是「輸入 + 新生成」整段，`decode` 全部還原。

2. **為什麼生成的內容寫到一半就停？**
   `max_new_tokens=20` 太小，達到上限就停。

3. **`.to(device)` 是在做什麼？**
   模型在 GPU，輸入張量也必須在 GPU。

4. **`model.generate` 和 `pipeline` 是什麼關係？**
   `pipeline` 在背後呼叫的就是 `model.generate`，再加上分詞、套用對話模板、還原與去掉提示詞。

5. **為什麼模型名稱要存成變數？**
   模型與分詞器要配對，用同一個變數就不會不一致。

6. **提示詞裡的空白有影響嗎？**
   三重引號會保留換行與縮排，這些都會被分詞，成為輸入的一部分。

## 六、常見錯誤排查表

| 症狀 | 可能原因 | 處理 |
|---|---|---|
| `CUDA out of memory` | 模型約 7.6 GB 顯存不足 | 改用 4 位元量化載入（見實驗 5） |
| 輸入與模型不在同一個裝置的錯誤 | 張量沒搬到 GPU | 確認有 `.to(device)` |
| 把 `device` 設成 `"auto"` 後 `.to(device)` 出錯 | `"auto"` 不是合法的裝置名稱，只能用在 `device_map` | 搬張量時改用 `model.device`（我沒有實測，這是依兩者的用法推論的） |
| 輸出被截斷 | `max_new_tokens=20` 太小 | 調大 |
| 輸出一堆特殊符號 | `decode` 預設不隱藏 | 加 `skip_special_tokens=True` |
| 載入時卡很久 | 第一次下載模型 | 等待，之後從快取讀取 |
| `torch_dtype` 的警告 | 新版 transformers 改名為 `dtype` | 依警告改名即可 |

## 七、觀察到的幾個地方，僅供參考

1. **提示詞字串含多餘空白**
   三重引號保留了換行與縮排，成為輸入的一部分。

2. **提示詞格式不完整**
   只有 `<|assistant|>`，沒有 `<|user|>` 與 `<|end|>`。較穩妥的做法是用 `apply_chat_template`。

3. **輸出含提示詞與特殊符號**
   如果只要生成的內容，需要自己切掉提示詞並隱藏特殊符號。

4. **沒有傳 `attention_mask`**
   只傳了 `input_ids`，函式庫可能會出現關於 attention mask 的警告。單一提示詞通常不影響結果，我沒有確認你的版本。

5. **沒有指定生成策略**
   `model.generate` 沒有指定 `do_sample`，使用模型的預設設定。

6. **`max_new_tokens=20` 很短**
   適合快速測試，看不到完整的回答。

7. **顯存**
   Phi-3-mini 以 16 位元載入，權重約 7.6 GB，8 GB 的顯卡可能放不下。

8. **`torch_dtype` 在新版可能改名為 `dtype`**。

## 八、可以自己做的實驗（需要時再自己加，不影響原程式）

```python
# 1. 看每一步的中間結果
print(input_ids)
print(input_ids.shape)                                     # (1, N)
print(tokenizer.convert_ids_to_tokens(input_ids[0]))       # 看提示詞被切成哪些 token

# 2. 看生成結果的形狀（會比 input_ids 長）
print(generation_output.shape)                             # (1, N + 20)

# 3. 只取「新生成」的部分，不含提示詞
new_tokens = generation_output[0][input_ids.shape[1]:]
print(tokenizer.decode(new_tokens, skip_special_tokens=True))

# 4. 隱藏特殊符號
print(tokenizer.decode(generation_output[0], skip_special_tokens=True))

# 5. 用 4 位元量化載入，大幅降低記憶體（需要 bitsandbytes 與 NVIDIA GPU）
from transformers import BitsAndBytesConfig
import torch
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.bfloat16)
model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb, device_map="auto")

# 6. 把提示詞的多餘空白去掉
import textwrap
prompt = textwrap.dedent("""
    Write an email apologizing to Jim for the trajic gardening mishap.
    Explain how it happened. <|assistant|>
""").strip()

# 7. 改用對話模板，由分詞器產生正確格式（比手動寫 <|assistant|> 穩妥）
messages = [{"role": "user", "content": "Write an email apologizing to Jim for the gardening mishap."}]
input_ids = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, return_tensors="pt"
).to(device)

# 8. 生成更長的回答
generation_output = model.generate(input_ids=input_ids, max_new_tokens=200)

# 9. 打開取樣，每次得到不同的結果
generation_output = model.generate(input_ids=input_ids, max_new_tokens=100,
                                   do_sample=True, temperature=0.8)
```


## 一句話記住

> **生成 = 分詞 → 模型生成 → 還原。** 一般寫法把這三步依序寫下來，每一步的結果都存在變數裡，你可以隨時印出來檢查。